# Generative Deep Learning: Autoencoders, VAEs, GANs & Diffusion Models

This interactive lab covers:
1. **Variational Autoencoder (VAE)**: Reparameterization trick, Gaussian latent space, and ELBO optimization.
2. **Generative Adversarial Network (GAN)**: Adversarial minimax optimization between Generator and Discriminator.
3. **Denoising Diffusion Probabilistic Models (DDPM)**: Markov noise schedule and closed-form forward sampling.

In [ ]:
import sys
import math
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import torch
import torch.nn as nn
import torch.optim as optim
from generative_engine import (
    VanillaAutoencoder, VariationalAutoencoder,
    GANGenerator, GANDiscriminator, DDPMNoiseSchedule
)

print('Generative Models modules loaded successfully!')

## 1. Variational Autoencoder (VAE) Training on Multi-Modal Data

We train a VAE with the reparameterization trick to compress a 16-dimensional multi-modal distribution into a 2D continuous Gaussian latent manifold.

In [ ]:
torch.manual_seed(42)
vae = VariationalAutoencoder(in_dim=16, hidden_dim=16, latent_dim=2)
optimizer = optim.AdamW(vae.parameters(), lr=0.01)

# Synthetic clustered data
X_train = torch.randn(200, 16) + torch.randint(0, 3, (200, 1)).float() * 3.0

history = []
for epoch in range(100):
    optimizer.zero_grad()
    recon, mu, logvar = vae(X_train)
    loss_dict = vae.elbo_loss(recon, X_train, mu, logvar)
    loss_dict['loss'].backward()
    optimizer.step()
    history.append((loss_dict['recon_loss'].item(), loss_dict['kl_div'].item()))

print(f'VAE Initial Recon Loss: {history[0][0]:.4f} | KL: {history[0][1]:.4f}')
print(f'VAE Final   Recon Loss: {history[-1][0]:.4f} | KL: {history[-1][1]:.4f}')
assert history[-1][0] < history[0][0], 'Reconstruction loss did not decrease!'

## 2. Generative Adversarial Network (GAN) Adversarial Step

We demonstrate the simultaneous optimization of the Generator and Discriminator.

In [ ]:
latent_dim = 8
data_dim = 4
gen = GANGenerator(latent_dim=latent_dim, out_dim=data_dim)
disc = GANDiscriminator(in_dim=data_dim)

opt_g = optim.Adam(gen.parameters(), lr=0.002)
opt_d = optim.Adam(disc.parameters(), lr=0.002)
criterion = nn.BCELoss()

# Single training step
real_samples = torch.randn(16, data_dim)
labels_real = torch.ones(16, 1)
labels_fake = torch.zeros(16, 1)

# Train Discriminator
opt_d.zero_grad()
loss_d_real = criterion(disc(real_samples), labels_real)
z = torch.randn(16, latent_dim)
fake_samples = gen(z)
loss_d_fake = criterion(disc(fake_samples.detach()), labels_fake)
loss_d = loss_d_real + loss_d_fake
loss_d.backward()
opt_d.step()

# Train Generator
opt_g.zero_grad()
loss_g = criterion(disc(fake_samples), labels_real)
loss_g.backward()
opt_g.step()

print(f'Discriminator Loss: {loss_d.item():.4f}')
print(f'Generator Loss:     {loss_g.item():.4f}')

## 3. Denoising Diffusion Probabilistic Models (DDPM) Forward Noise Process

We track how the signal-to-noise ratio (SNR) decays across 50 diffusion timesteps via closed-form sampling $q(x_t \mid x_0)$.

In [ ]:
schedule = DDPMNoiseSchedule(timesteps=50, beta_start=1e-4, beta_end=0.02)
x_0 = torch.ones(1, 4) * 5.0  # Constant signal vector

print(f'{"Timestep (t)":<15} | {"Alpha_bar_t":<15} | {"Signal Weight":<16} | {"Noise Weight":<16}')
print('-' * 66)
sample_steps = [0, 10, 25, 40, 49]
for t_step in sample_steps:
    t_tensor = torch.tensor([t_step])
    alpha_bar = schedule.alphas_cumprod[t_step].item()
    signal_w = math.sqrt(alpha_bar)
    noise_w = math.sqrt(1.0 - alpha_bar)
    print(f'{t_step:<15} | {alpha_bar:<15.4f} | {signal_w:<16.4f} | {noise_w:<16.4f}')